In [1]:
import pandas as pd
import numpy as np
import io

# ============================================================
# LABORATOR 7 — VERIFICAREA CALITĂȚII DATELOR ȘI TRATAREA VALORILOR
# Stil: variabile în română, pași numerotați, parametri la început
# ============================================================

# ------------------------------------------------------------
# CARTEA DE IDENTITATE A VERIFICĂRII (parametri, nu cifre în cod)
# ------------------------------------------------------------
PARAMETRI_VERIFICARE = {
    'scop':                    'Verificarea plauzibilității și tratarea valorilor aberante/lipsă',
    'variabila':               'valoare',
    'interval_plauzibilitate': (0, 1000),   # (min, max) acceptat
    'regula_iqr':              1.5,          # coeficientul k din regula IQR
    'tratament':               'corectare / păstrare+marcare / păstrare+semnalare',
    'prag_cv':                 30.0,         # % — peste => variabilitate mare
    'prag_completitudine':     90.0,         # % — sub => publicăm cu rezerve
    'prag_imputare':           10.0,         # % — peste => rezerve
}

# Datele de intrare (30 de valori, 3 grupe, 4 lipsă, 3 semnalate IQR)
# Înlocuiește cu citirea din fișier: pd.read_csv("lab7.csv")
date_brute_lab7 = pd.DataFrame({
    'id':     range(1, 31),
    'grup':   ['G1']*10 + ['G2']*10 + ['G3']*10,
    'valoare': [
        12, 14, 15, 13, 16, 14, 15, np.nan, 13, 14,          # G1 (1 lipsă)
        20, 22, 21, np.nan, 23, 24, 22, 21, 20, np.nan,      # G2 (2 lipsă)
        30, 31, 32, 33, np.nan, 35, 36, 37, 900, 38,          # G3 (1 lipsă + 900 aberant)
    ],
})

# ============================================================
# PASUL 0 — Citirea / pregătirea datelor
# ============================================================
cadru_date_lab7 = date_brute_lab7.copy()
cadru_date_lab7['valoare'] = pd.to_numeric(cadru_date_lab7['valoare'], errors='coerce')

print("=" * 60)
print("PAS 0 — Date încărcate")
print("=" * 60)
print(cadru_date_lab7.head(10).to_string(index=False))
print(f"Total înregistrări: {len(cadru_date_lab7)}")


# ============================================================
# PASUL 1 — Inventar: observate, lipsă, neplauzibile
# ============================================================
total_inregistrari = len(cadru_date_lab7)
valori_lipsa      = cadru_date_lab7['valoare'].isna().sum()
valori_observate  = total_inregistrari - valori_lipsa

min_plauzibil, max_plauzibil = PARAMETRI_VERIFICARE['interval_plauzibilitate']
masca_neplauzibil = (
    cadru_date_lab7['valoare'].notna() &
    ~cadru_date_lab7['valoare'].between(min_plauzibil, max_plauzibil)
)
valori_neplauzibile = int(masca_neplauzibil.sum())

rata_raspuns       = 100 * valori_observate / total_inregistrari
pondere_lipsa      = 100 * valori_lipsa / total_inregistrari

print("\n" + "=" * 60)
print("PASUL 1 — Inventar")
print("=" * 60)
print(f"Total:                {total_inregistrari}")
print(f"Observate:            {valori_observate}")
print(f"Lipsă:                {valori_lipsa}")
print(f"Neplauzibile:         {valori_neplauzibile}")
print(f"Rata de răspuns:      {rata_raspuns:.1f}%")
print(f"Pondere valori lipsă: {pondere_lipsa:.1f}%")


# ============================================================
# PASUL 2 — Aberante: Q1, Q3, IQR, limite, valori semnalate
# ============================================================
k_iqr = PARAMETRI_VERIFICARE['regula_iqr']
valori_observate_serie = cadru_date_lab7['valoare'].dropna().sort_values().reset_index(drop=True)

q1  = valori_observate_serie.quantile(0.25)
q3  = valori_observate_serie.quantile(0.75)
iqr = q3 - q1
limita_inferioara = q1 - k_iqr * iqr
limita_superioara = q3 + k_iqr * iqr

masca_aberant = (
    cadru_date_lab7['valoare'].notna() &
    ((cadru_date_lab7['valoare'] < limita_inferioara) |
     (cadru_date_lab7['valoare'] > limita_superioara))
)
valori_semnalate = cadru_date_lab7[masca_aberant].copy()
valori_semnalate['pozitie'] = valori_semnalate.index + 1

print("\n" + "=" * 60)
print("PASUL 2 — Valori aberante (regula IQR)")
print("=" * 60)
print(f"Q1 = {q1:.2f} | Q3 = {q3:.2f} | IQR = {iqr:.2f} | k = {k_iqr}")
print(f"Limite: [{limita_inferioara:.2f}, {limita_superioara:.2f}]")
print("\nValori semnalate:")
print(valori_semnalate[['pozitie', 'id', 'grup', 'valoare']].to_string(index=False))


# ============================================================
# PASUL 3 — Cazuri: decizie per valoare semnalată
# Deciziile din fișă (exemplu): corectare, păstrare+marcare, păstrare+semnalare
# ============================================================
# Dicționar decizii: id -> (decizie, valoare_corectata_sau_None)
DECIZII_CAZURI = {
    # exemplu: 900 -> corectare la 39 (valoare tipică G3)
    29: ('corectare', 39.0),
}
# toate celelalte valori semnalate rămân „păstrare + semnalare"
cadru_date_lab7['decizie']         = 'păstrat'
cadru_date_lab7['valoare_tratata'] = cadru_date_lab7['valoare']

for idx, rand in valori_semnalate.iterrows():
    id_rand = rand['id']
    if id_rand in DECIZII_CAZURI:
        decizie, val_noua = DECIZII_CAZURI[id_rand]
        cadru_date_lab7.at[idx, 'decizie'] = decizie
        if decizie == 'corectare' and val_noua is not None:
            cadru_date_lab7.at[idx, 'valoare_tratata'] = val_noua
    else:
        cadru_date_lab7.at[idx, 'decizie'] = 'păstrare+semnalare'

print("\n" + "=" * 60)
print("PASUL 3 — Decizii pe cazuri")
print("=" * 60)
print(cadru_date_lab7.loc[masca_aberant, ['id', 'grup', 'valoare', 'decizie', 'valoare_tratata']]
      .to_string(index=False))

# Media și mediana înainte/după, cu și fără valoarea neconfirmată
def medii_median(grup_serie):
    return grup_serie.mean(), grup_serie.median()

media_bruta, mediana_bruta       = medii_median(cadru_date_lab7['valoare'].dropna())
media_tratata, mediana_tratata   = medii_median(cadru_date_lab7['valoare_tratata'].dropna())

# fără valoarea neconfirmată (900 exclus)
fara_neconfirmat = cadru_date_lab7.loc[~masca_aberant, 'valoare_tratata'].dropna()
media_fara, mediana_fara = medii_median(fara_neconfirmat)

print(f"\nMedia   înainte: {media_bruta:.2f}  | după: {media_tratata:.2f} | fără neconfirmat: {media_fara:.2f}")
print(f"Mediana înainte: {mediana_bruta:.2f}  | după: {mediana_tratata:.2f} | fără neconfirmat: {mediana_fara:.2f}")


# ============================================================
# PASUL 4 — Imputare: mediana grupului, marcare, rata de imputare
# ============================================================
cadru_date_lab7['valoare_finala'] = cadru_date_lab7['valoare_tratata']
mediane_grup = cadru_date_lab7.groupby('grup')['valoare_tratata'].transform('median')

masca_lipsa = cadru_date_lab7['valoare_finala'].isna()
cadru_date_lab7.loc[masca_lipsa, 'valoare_finala'] = mediane_grup[masca_lipsa]
cadru_date_lab7['imputat'] = masca_lipsa

rata_imputare = 100 * masca_lipsa.sum() / total_inregistrari

print("\n" + "=" * 60)
print("PASUL 4 — Imputare cu mediana grupului")
print("=" * 60)
print(f"Valori imputate: {int(masca_lipsa.sum())} ({rata_imputare:.1f}%)")
print(cadru_date_lab7.loc[masca_lipsa, ['id', 'grup', 'valoare', 'valoare_finala', 'imputat']]
      .to_string(index=False))


# ============================================================
# PASUL 5 — Indicatori: media, s, SE, CV, efectul tratării
# ============================================================
serie_finala = cadru_date_lab7['valoare_finala'].dropna()
media_finala    = serie_finala.mean()
dev_std_finala  = serie_finala.std(ddof=1)
n_finala        = serie_finala.count()
se_finala       = dev_std_finala / np.sqrt(n_finala)
cv_finala       = 100 * dev_std_finala / media_finala
efect_medie     = media_finala - media_bruta

prag_cv = PARAMETRI_VERIFICARE['prag_cv']
verdict_cv = "OK" if cv_finala <= prag_cv else "CV mare"

print("\n" + "=" * 60)
print("PASUL 5 — Indicatori finali")
print("=" * 60)
print(f"n:        {n_finala}")
print(f"Media:    {media_finala:.2f}")
print(f"s:        {dev_std_finala:.2f}")
print(f"SE:       {se_finala:.2f}")
print(f"CV:       {cv_finala:.1f}%  (prag {prag_cv}% -> {verdict_cv})")
print(f"Efectul tratării asupra mediei: {efect_medie:+.2f}")


# ============================================================
# PASUL 6 — Raport: tabel indicator – valoare – prag – verdict
# ============================================================
prag_completitudine = PARAMETRI_VERIFICARE['prag_completitudine']
prag_imputare       = PARAMETRI_VERIFICARE['prag_imputare']

randuri_raport = [
    ('Rata de răspuns',      f"{rata_raspuns:.1f}%",       f"≥ {prag_completitudine}%",
     "OK" if rata_raspuns >= prag_completitudine else "Rezerve"),
    ('Pondere lipsă',        f"{pondere_lipsa:.1f}%",       f"≤ {100 - prag_completitudine:.1f}%",
     "OK" if pondere_lipsa <= 100 - prag_completitudine else "Rezerve"),
    ('Valori neplauzibile',  f"{valori_neplauzibile}",      "0",
     "OK" if valori_neplauzibile == 0 else "Rezerve"),
    ('Valori aberante IQR',  f"{len(valori_semnalate)}",    "tratate",
     "OK"),
    ('Rata de imputare',     f"{rata_imputare:.1f}%",       f"≤ {prag_imputare}%",
     "OK" if rata_imputare <= prag_imputare else "Rezerve"),
    ('CV',                   f"{cv_finala:.1f}%",           f"≤ {prag_cv}%",
     verdict_cv),
    ('Efect tratare medie',  f"{efect_medie:+.2f}",         "mic",
     "OK" if abs(efect_medie) < 0.1 * abs(media_bruta) else "Semnificativ"),
]

tabel_raport = pd.DataFrame(randuri_raport, columns=['Indicator', 'Valoare', 'Prag', 'Verdict'])

print("\n" + "=" * 60)
print("PASUL 6 — Raport de calitate")
print("=" * 60)
print(tabel_raport.to_string(index=False))

# Decizia finală
are_rezerve = (tabel_raport['Verdict'] == 'Rezerve').any() or (tabel_raport['Verdict'] == 'CV mare').any()
decizie_finala = "PUBLICĂM CU REZERVE" if are_rezerve else "PUBLICĂM"
print(f"\nDecizie finală: {decizie_finala}")

# Salvare raport în CSV
tabel_raport.to_csv("raport_lab7.csv", index=False)
print("Raport salvat în: raport_lab7.csv")


# ============================================================
# EXERCIȚIUL 1 — Extinde intervalul de plauzibilitate și compară
# ============================================================
print("\n" + "=" * 60)
print("EXERCIȚIUL 1 — Interval extins de plauzibilitate")
print("=" * 60)
# ... cod ...
# Sugestie: schimbă PARAMETRI_VERIFICARE['interval_plauzibilitate'] = (0, 10000)
# și rulează din nou pașii 1-2; observă câte valori mai rămân neplauzibile.


# ============================================================
# EXERCIȚIUL 2 — Imputare cu media grupului în loc de mediană
# ============================================================
print("\n" + "=" * 60)
print("EXERCIȚIUL 2 — Imputare cu media grupului")
print("=" * 60)
# ... cod ...
# Sugestie: înlocuiește 'median' cu 'mean' în transform() și compară rata/efectul.


# ============================================================
# EXERCIȚIUL 3 — Alt coeficient IQR (k = 3.0) — valori extreme
# ============================================================
print("\n" + "=" * 60)
print("EXERCIȚIUL 3 — Regula IQR cu k = 3.0")
print("=" * 60)
# ... cod ...
# Sugestie: schimbă PARAMETRI_VERIFICARE['regula_iqr'] = 3.0 și rulează pasul 2.


# ============================================================
# EXERCIȚIUL 4 — Raport pe fiecare grup separat
# ============================================================
print("\n" + "=" * 60)
print("EXERCIȚIUL 4 — Indicatori pe grup")
print("=" * 60)
# ... cod ...
# Sugestie: groupby('grup') pe valoare_finala -> media, s, CV; apoi compară cu pragul.

PAS 0 — Date încărcate
 id grup  valoare
  1   G1     12.0
  2   G1     14.0
  3   G1     15.0
  4   G1     13.0
  5   G1     16.0
  6   G1     14.0
  7   G1     15.0
  8   G1      NaN
  9   G1     13.0
 10   G1     14.0
Total înregistrări: 30

PASUL 1 — Inventar
Total:                30
Observate:            26
Lipsă:                4
Neplauzibile:         0
Rata de răspuns:      86.7%
Pondere valori lipsă: 13.3%

PASUL 2 — Valori aberante (regula IQR)
Q1 = 15.00 | Q3 = 31.75 | IQR = 16.75 | k = 1.5
Limite: [-10.12, 56.88]

Valori semnalate:
 pozitie  id grup  valoare
      29  29   G3    900.0

PASUL 3 — Decizii pe cazuri
 id grup  valoare   decizie  valoare_tratata
 29   G3    900.0 corectare             39.0

Media   înainte: 56.58  | după: 23.46 | fără neconfirmat: 22.84
Mediana înainte: 21.50  | după: 21.50 | fără neconfirmat: 21.00

PASUL 4 — Imputare cu mediana grupului
Valori imputate: 4 (13.3%)
 id grup  valoare  valoare_finala  imputat
  8   G1      NaN            14.0     T